# Pipeline Integration Test

**Purpose:**
Validate the three upstream classifiers end-to-end before Module 4 RAG is connected.

Each section tests one module in isolation then Section 5 runs a full pipeline simulation feeding all three results together.



## 1. Environment Setup

In [1]:
!pip install torch transformers groq python-dotenv joblib

In [2]:
import warnings
warnings.filterwarnings('ignore')

import os
import json
import time
import joblib
import torch
import numpy as np
import pandas as pd
from pathlib import Path
from typing  import Optional
from dotenv  import load_dotenv
from groq    import Groq
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification
)

In [3]:
load_dotenv()
GROQ_API_KEY = os.getenv('GROQ_API_KEY')
HF_MODEL_NAME = os.getenv('HF_MODEL_NAME')

assert GROQ_API_KEY,  "GROQ_API_KEY missing from .env"
assert HF_MODEL_NAME, "HF_MODEL_NAME missing from .env"

# Artifact paths (relative to this notebook in pipeline/)
M1_ARTIFACT = Path('../module_1_language_detection/language_detector.joblib')
M3_ARTIFACT = Path('../module_3_intent_classifier/artifacts')

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f"GROQ_API_KEY  : {'set' if GROQ_API_KEY else 'MISSING'}")
print(f"HF_MODEL_NAME : {HF_MODEL_NAME}")
print(f"M1 artifact   : {M1_ARTIFACT}  (exists: {M1_ARTIFACT.exists()})")
print(f"M3 artifacts  : {M3_ARTIFACT}  (exists: {M3_ARTIFACT.exists()})")
print(f"Device        : {DEVICE}")

GROQ_API_KEY  : set
HF_MODEL_NAME : HagarGhazi/emotion-classifier-mental-health
M1 artifact   : ..\module_1_language_detection\language_detector.joblib  (exists: True)
M3 artifacts  : ..\module_3_intent_classifier\artifacts  (exists: True)
Device        : cpu


## 2. Language Detection Module

Loads language_detector.joblib which bundles the TF-IDF vectorizer and the best classical ML model.

The detect_language() function is reproduced here exactly as it was defined in Module 1 so that inference behavior is identical.

In [4]:
# Load the inference pipeline saved in Module 1
m1 = joblib.load(M1_ARTIFACT)
m1_model      = m1['model']
m1_vectorizer = m1['vectorizer']
LANGUAGE_META       = m1['language_meta']
CONFIDENCE_THRESHOLD = m1['confidence_threshold']
SHORT_THRESHOLD      = m1['short_threshold']


print(f"Model              : {m1['model_name']}")
print(f"Val accuracy       : {m1['val_accuracy']:.4f}")
print(f"Test accuracy      : {m1['test_accuracy']:.4f}")
print(f"Vocabulary size    : {len(m1_vectorizer.vocabulary_):,}")
print(f"Languages          : {len(LANGUAGE_META)}")
print(f"Confidence threshold : {CONFIDENCE_THRESHOLD}")
print(f"Short threshold      : {SHORT_THRESHOLD} chars")

Model              : LinearSVC
Val accuracy       : 0.9955
Test accuracy      : 0.9953
Vocabulary size    : 300,000
Languages          : 20
Confidence threshold : 0.6
Short threshold      : 15 chars


In [5]:
def preprocess_text(text: str) -> str:
    return ' '.join(text.strip().split())                          # Normalize whitespace only as character n-grams must see punctuation and accents


def detect_language(text: str, model, vectorizer, threshold: float = CONFIDENCE_THRESHOLD) -> dict:
    clean      = preprocess_text(text)
    is_short   = len(clean) <= SHORT_THRESHOLD
    features   = vectorizer.transform([clean])
    prediction = model.predict(features)[0]
    proba      = model.predict_proba(features)[0]
    confidence = proba.max()
    classes    = model.classes_
    top3_idx = proba.argsort()[-3:][::-1]
    top3     = [(classes[i], proba[i]) for i in top3_idx]
    trusted = not (is_short and confidence < threshold)            # Distrust very short texts that fall below the confidence threshold


    return {
        'text'       : text,
        'prediction' : prediction,
        'lang_name'  : LANGUAGE_META[prediction][0],
        'confidence' : confidence,
        'trusted'    : trusted,
        'is_short'   : is_short,
        'top3'       : top3
    }


print("detect_language() ready")

detect_language() ready


### 2.1 Functional Smoke Test

One sample per language family to confirm the artifact loads and predicts correctly before the bulk test.

In [6]:
smoke_tests = [
    ('en', 'I have been feeling very anxious and overwhelmed lately'),
    ('ar', 'أشعر بقلق شديد ولا أعرف ماذا أفعل'),
    ('de', 'Ich fühle mich in letzter Zeit sehr ängstlich'),
    ('zh', '我最近感到非常焦虑，不知道该怎么办'),
    ('ru', 'В последнее время я чувствую сильное беспокойство'),
    ('fr', 'Je me sens très anxieux ces derniers temps'),
    ('hi', 'मुझे हाल ही में बहुत चिंता हो रही है'),
    ('ja', '最近、非常に不安を感じています'),
]

print("Language Detection Smoke Test\n")
correct = 0
for true_lang, text in smoke_tests:
    r      = detect_language(text, m1_model, m1_vectorizer)
    status = 'OK' if r['prediction'] == true_lang else 'FAIL'
    if r['prediction'] == true_lang:
        correct += 1
    print(f"  [{status}] True: {true_lang} ({LANGUAGE_META[true_lang][0]:<12}) | Pred: {r['prediction']} ({r['lang_name']:<12}) | Conf: {r['confidence']:.3f}")

print(f"\nSmoke test accuracy : {correct}/{len(smoke_tests)} ({correct/len(smoke_tests)*100:.0f}%)")

Language Detection Smoke Test

  [OK] True: en (English     ) | Pred: en (English     ) | Conf: 0.980
  [OK] True: ar (Arabic      ) | Pred: ar (Arabic      ) | Conf: 0.987
  [OK] True: de (German      ) | Pred: de (German      ) | Conf: 0.995
  [OK] True: zh (Chinese     ) | Pred: zh (Chinese     ) | Conf: 0.983
  [OK] True: ru (Russian     ) | Pred: ru (Russian     ) | Conf: 0.992
  [OK] True: fr (French      ) | Pred: fr (French      ) | Conf: 0.982
  [OK] True: hi (Hindi       ) | Pred: hi (Hindi       ) | Conf: 0.994
  [OK] True: ja (Japanese    ) | Pred: ja (Japanese    ) | Conf: 0.978

Smoke test accuracy : 8/8 (100%)


### 2.2 Full Batch Test

Covers all 20 languages with both long and short inputs including the hard pairs ar <---> ur and bg <---> ru.

In [7]:
m1_test_cases = [
    # Long sentences
    ('en', 'Mental health is a critical component of overall well-being that affects how we think, feel, and act.'),
    ('ar', 'الصحة النفسية جزء أساسي من الصحة العامة وتؤثر على طريقة تفكيرنا ومشاعرنا وسلوكنا في الحياة اليومية.'),
    ('ur', 'ذہنی صحت مجموعی صحت کا ایک اہم حصہ ہے جو ہماری سوچ، احساسات اور رویے کو متاثر کرتی ہے۔'),
    ('de', 'Die psychische Gesundheit ist ein wesentlicher Bestandteil des allgemeinen Wohlbefindens.'),
    ('fr', 'La santé mentale est un élément crucial du bien-être général qui affecte notre façon de penser.'),
    ('es', 'La salud mental es un componente crítico del bienestar general y afecta cómo pensamos y actuamos.'),
    ('pt', 'A saúde mental é um componente crítico do bem-estar geral e afeta como pensamos e agimos.'),
    ('ru', 'Психическое здоровье является важнейшим компонентом общего благополучия человека.'),
    ('bg', 'Психичното здраве е съществена част от общото благосъстояние на човека.'),
    ('zh', '心理健康是整体健康的重要组成部分，影响我们的思维、感受和行为方式。'),
    ('ja', '精神的な健康は全体的な幸福の重要な要素であり、私たちの思考、感情、行動に影響します。'),
    ('hi', 'मानसिक स्वास्थ्य समग्र कल्याण का एक महत्वपूर्ण घटक है जो हमारी सोच और व्यवहार को प्रभावित करता है।'),
    ('tr', 'Ruh sağlığı, genel refahın kritik bir bileşenidir ve nasıl düşündüğümüzü etkiler.'),
    ('it', 'La salute mentale è una componente critica del benessere generale e influenza il nostro modo di pensare.'),
    ('nl', 'Geestelijke gezondheid is een cruciaal onderdeel van het algemeen welzijn en beïnvloedt hoe wij denken.'),
    ('pl', 'Zdrowie psychiczne jest kluczowym elementem ogólnego dobrostanu i wpływa na nasze myślenie.'),
    ('vi', 'Sức khỏe tâm thần là một thành phần quan trọng của sức khỏe tổng thể và ảnh hưởng đến cách chúng ta suy nghĩ.'),
    ('th', 'สุขภาพจิตเป็นองค์ประกอบสำคัญของสุขภาพโดยรวมที่มีผลต่อวิธีที่เราคิด รู้สึก และกระทำ'),
    ('sw', 'Afya ya akili ni sehemu muhimu ya ustawi wa jumla na huathiri jinsi tunavyofikiri na kutenda.'),
    ('el', 'Η ψυχική υγεία είναι κρίσιμο συστατικό της συνολικής ευημερίας και επηρεάζει τον τρόπο που σκεφτόμαστε.'),

    # Short inputs
    ('en', 'I need help'),
    ('ar', 'أحتاج مساعدة'),
    ('ur', 'مجھے مدد چاہیے'),
    ('de', 'ich brauche Hilfe'),
    ('fr', "j'ai besoin d'aide"),
    ('es', 'necesito ayuda'),
    ('ru', 'мне нужна помощь'),
    ('bg', 'имам нужда от помощ'),
    ('zh', '我需要帮助'),
    ('ja', '助けてください'),

    # Hard confusion pairs
    ('pt', 'Eu estou sentindo muita ansiedade ultimamente e não consigo dormir direito.'),
    ('es', 'Estoy sintiendo mucha ansiedad últimamente y no puedo dormir bien.'),
    ('bg', 'Чувствам се много тревожен напоследък и не мога да спя добре.'),
    ('ru', 'В последнее время я чувствую сильное беспокойство и не могу нормально спать.'),
    ('ar', 'أشعر بقلق شديد في الآونة الأخيرة ولا أستطيع النوم جيداً.'),
    ('ur', 'میں حال ہی میں بہت پریشان محسوس کر رہا ہوں اور ٹھیک سے سو نہیں پا رہا۔'),
]


In [8]:
m1_records = []
for true_lang, text in m1_test_cases:
    r = detect_language(text, m1_model, m1_vectorizer)
    m1_records.append({
        'true'      : true_lang,
        'pred'      : r['prediction'],
        'correct'   : r['prediction'] == true_lang,
        'confidence': round(r['confidence'], 4),
        'trusted'   : r['trusted'],
        'is_short'  : r['is_short'],
        'text'      : text[:65]
    })


m1_df = pd.DataFrame(m1_records)
m1_acc_overall = m1_df['correct'].mean()
m1_acc_long    = m1_df[~m1_df['is_short']]['correct'].mean()
m1_acc_short   = m1_df[m1_df['is_short']]['correct'].mean()

print(f"Batch Test Results")
print()
print(f"  Overall accuracy : {m1_acc_overall:.4f} ({m1_df['correct'].sum()}/{len(m1_df)})")
print(f"  Long inputs      : {m1_acc_long:.4f}")
print(f"  Short inputs     : {m1_acc_short:.4f}")
print(f"  Mean confidence  : {m1_df['confidence'].mean():.4f}")
print()

pd.set_option('display.max_colwidth', 70)
print(m1_df.to_string(index = False))

Batch Test Results

  Overall accuracy : 1.0000 (36/36)
  Long inputs      : 1.0000
  Short inputs     : 1.0000
  Mean confidence  : 0.9723

true pred  correct  confidence  trusted  is_short                                                              text
  en   en     True      0.9878     True     False Mental health is a critical component of overall well-being that 
  ar   ar     True      0.9938     True     False الصحة النفسية جزء أساسي من الصحة العامة وتؤثر على طريقة تفكيرنا و
  ur   ur     True      0.9941     True     False ذہنی صحت مجموعی صحت کا ایک اہم حصہ ہے جو ہماری سوچ، احساسات اور ر
  de   de     True      0.9910     True     False Die psychische Gesundheit ist ein wesentlicher Bestandteil des al
  fr   fr     True      0.9813     True     False La santé mentale est un élément crucial du bien-être général qui 
  es   es     True      0.9973     True     False La salud mental es un componente crítico del bienestar general y 
  pt   pt     True      0.9851     True     Fal

### 2.3 Failure Analysis

Any incorrect predictions are surfaced here with their full top-3 breakdown so the confusion source is visible.

In [9]:
failures = [(true_lang, text) for (true_lang, text), rec in zip(m1_test_cases, m1_records) if not rec['correct']]

if not failures:
    print("No failures as all predictions correct.")
else:
    print(f"Failures ({len(failures)} total):\n")
    for true_lang, text in failures:
        r = detect_language(text, m1_model, m1_vectorizer)
        print(f"  True     : {true_lang} ({LANGUAGE_META[true_lang][0]})")
        print(f"  Text     : {text[:70]}")
        print(f"  Pred     : {r['prediction']} ({r['lang_name']})  confidence = {r['confidence']:.4f}")
        top3_str = ', '.join([f"{c} = {p:.3f}" for c, p in r['top3']])
        print(f"  Top 3    : {top3_str}")
        print()

No failures as all predictions correct.


## 3. Emotion Classifier Module

Loads the fine-tuned DistilBERT model from HuggingFace Hub using the HF_MODEL_NAME environment variable.  

The classify_emotion() function mirrors the one defined in Module 2 including the safety override for isolation phrases.

In [10]:
# Load tokenizer and model from HuggingFace Hub
hf_tokenizer = AutoTokenizer.from_pretrained(HF_MODEL_NAME)
hf_model     = AutoModelForSequenceClassification.from_pretrained(HF_MODEL_NAME)
hf_model     = hf_model.to(DEVICE)
hf_model.eval()
print(f"Model loaded : {HF_MODEL_NAME}")
print(f"Device       : {DEVICE}")

Model loaded : HagarGhazi/emotion-classifier-mental-health
Device       : cpu


In [11]:
EMOTION_META = {
    0: {'name': 'sadness',  'mh_significance': 'Depression signal — highest clinical risk',      'response_tone': 'Warm, validating, gentle — avoid minimizing'},
    1: {'name': 'joy',      'mh_significance': 'Positive state — reinforce and sustain',         'response_tone': 'Encouraging, celebratory, engaging'},
    2: {'name': 'love',     'mh_significance': 'Connection-seeking — social support signal',     'response_tone': 'Warm, affirming, relationship-focused'},
    3: {'name': 'anger',    'mh_significance': 'Frustration or distress — needs de-escalation',  'response_tone': 'Calm, empathetic, non-confrontational'},
    4: {'name': 'fear',     'mh_significance': 'Anxiety signal — very common in mental health',  'response_tone': 'Reassuring, grounding, structured'},
    5: {'name': 'surprise', 'mh_significance': 'Neutral-to-positive — low clinical risk',        'response_tone': 'Curious, engaged, open'},
}

LABEL_TO_NAME = {k: v['name'] for k, v in EMOTION_META.items()}
NAME_TO_LABEL = {v['name']: k for k, v in EMOTION_META.items()}

ISOLATION_PHRASES = [
    'nobody understands', 'no one understands',
    'nobody cares', 'no one cares',
    'all alone', 'completely alone',
    'nobody listens', 'no one listens'
]


def classify_emotion(text: str, confidence_threshold: float = 0.40) -> dict:
    text_lower = text.lower()

    # Safety override before model inference
    if any(phrase in text_lower for phrase in ISOLATION_PHRASES):
        return {
            'emotion'         : 'sadness',
            'confidence'      : 1.0,
            'mh_significance' : 'Depression signal — highest clinical risk',
            'response_tone'   : 'Warm, validating, gentle — avoid minimizing',
            'risk_flag'       : True,
            'llm_instruction' : (
                'The user appears to be experiencing sadness and isolation. '
                'Warm, validating, gentle — avoid minimizing. '
                '⚠ High-risk emotional state — prioritize safety and validation above all.'
            )
        }

    inputs = hf_tokenizer(
        text[:512],
        max_length      = 128,
        padding         = 'max_length',
        truncation      = True,
        return_tensors  = 'pt',
        return_token_type_ids = False    # DistilBERT doesn't use this
    )
    inputs = {k: v.to(DEVICE) for k, v in inputs.items()}

    hf_model.eval()
    with torch.no_grad():
        outputs = hf_model(**inputs)
        probs   = torch.softmax(outputs.logits, dim = -1).squeeze()

    scores     = {i: probs[i].item() for i in range(len(probs))}
    top_label  = max(scores, key = scores.get)
    confidence = scores[top_label]
    label_name = LABEL_TO_NAME[top_label]

    meta = EMOTION_META.get(top_label, {})

    if confidence < confidence_threshold:
        return {
            'emotion'         : 'uncertain',
            'confidence'      : round(confidence, 4),
            'mh_significance' : 'Ambiguous — treat as neutral',
            'response_tone'   : 'Open, curious, non-assumptive',
            'risk_flag'       : False,
            'llm_instruction' : 'Treat as emotionally neutral until more context is available.'
        }

    is_high_risk = label_name in ('sadness', 'fear') and confidence > 0.80

    return {
        'emotion'         : label_name,
        'confidence'      : round(confidence, 4),
        'mh_significance' : meta.get('mh_significance', ''),
        'response_tone'   : meta.get('response_tone', ''),
        'risk_flag'       : is_high_risk,
        'llm_instruction' : (
            f'The user appears to be experiencing {label_name}. '
            f"{meta.get('response_tone', '')}. "
            + ('High-risk emotional state — prioritize safety and validation above all.' if is_high_risk else '')
        )
    }


print("classify_emotion() ready")

classify_emotion() ready


### 3.1 Functional Smoke Test

One representative sample per emotion class plus two multilingual inputs.

In [12]:
smoke_emotions = [
    ('sadness',  'i feel so worthless and i do not know what to do anymore'),
    ('joy',      'i am so happy and grateful for everything in my life right now'),
    ('love',     'i feel so deeply connected to everyone around me'),
    ('anger',    'i am so frustrated and angry at everything and everyone'),
    ('fear',     'i am terrified about what is going to happen to me'),
    ('surprise', 'i am completely shocked by what just happened'),
    ('sadness',  'nobody understands me and i am so alone'),    # isolation phrase override
    ('fear',     'je me sens très anxieux'),                    # French
    ('sadness',  'مجھے بہت اداسی ہو رہی ہے'),                  # Urdu
]


In [13]:
print("Smoke Test\n")
for expected, text in smoke_emotions:
    r      = classify_emotion(text)
    status = 'OK' if r['emotion'] == expected else 'FAIL'
    flag   = ' [HIGH RISK]' if r['risk_flag'] else ''
    print(f"  [{status}] Expected: {expected:<10} | Pred: {r['emotion']:<10} | Conf: {r['confidence']:.3f}{flag}")
    print(f"         Text        : {text[:70]}")
    print()

Smoke Test

  [OK] Expected: sadness    | Pred: sadness    | Conf: 1.000 [HIGH RISK]
         Text        : i feel so worthless and i do not know what to do anymore

  [OK] Expected: joy        | Pred: joy        | Conf: 0.997
         Text        : i am so happy and grateful for everything in my life right now

  [FAIL] Expected: love       | Pred: joy        | Conf: 0.873
         Text        : i feel so deeply connected to everyone around me

  [OK] Expected: anger      | Pred: anger      | Conf: 0.998
         Text        : i am so frustrated and angry at everything and everyone

  [OK] Expected: fear       | Pred: fear       | Conf: 0.996 [HIGH RISK]
         Text        : i am terrified about what is going to happen to me

  [OK] Expected: surprise   | Pred: surprise   | Conf: 1.000
         Text        : i am completely shocked by what just happened

  [OK] Expected: sadness    | Pred: sadness    | Conf: 1.000 [HIGH RISK]
         Text        : nobody understands me and i am so 

### 3.2 Full Batch Test

Covers all six emotion classes, high-risk flag activation, the safety override for isolation phrases,
and multilingual inputs that the model was designed to handle.

In [14]:
m2_test_cases = [
    # Sadness —> depression signal
    ('sadness', 'i feel so empty and hopeless i cannot imagine things ever getting better'),
    ('sadness', 'i cry every night and i do not even know why anymore'),
    ('sadness', 'i feel like a burden to everyone around me'),
    ('sadness', 'i have lost all interest in things i used to love'),
    ('sadness', 'nobody understands me and i feel completely alone'),      # isolation override

    # Joy —> positive state
    ('joy', 'i am so excited about my future and feel full of energy'),
    ('joy', 'today was wonderful and i feel genuinely happy'),
    ('joy', 'i finally feel like things are coming together in my life'),

    # Love —> connection seeking
    ('love', 'i feel so much love and warmth for the people in my life'),
    ('love', 'i am deeply grateful for all the support i receive'),

    # Anger —> frustration / distress
    ('anger', 'i am so angry at how unfair everything is'),
    ('anger', 'i cannot control my rage and it scares me'),
    ('anger', 'i feel furious and want to push everyone away'),

    # Fear —> anxiety signal
    ('fear', 'i am so scared about the future and i cannot stop worrying'),
    ('fear', 'i have panic attacks every morning before work'),
    ('fear', 'i feel constantly on edge like something terrible is about to happen'),

    # Surprise —> neutral
    ('surprise', 'i was completely shocked by the news today'),
    ('surprise', 'i cannot believe how quickly everything changed'),

    # Multilingual inputs
    ('sadness', 'je me sens très triste et seul ces derniers temps'),        # French
    ('fear',    'estoy muy asustado y no sé cómo manejar esta ansiedad'),    # Spanish
    ('sadness', 'ich fühle mich so leer und hoffnungslos'),                  # German
]

In [15]:
m2_records = []
for expected_emotion, text in m2_test_cases:
    r = classify_emotion(text)
    m2_records.append({
        'expected'   : expected_emotion,
        'pred'       : r['emotion'],
        'correct'    : r['emotion'] == expected_emotion,
        'confidence' : r['confidence'],
        'risk_flag'  : r['risk_flag'],
        'text'       : text[:65]
    })


m2_df = pd.DataFrame(m2_records)
m2_acc = m2_df['correct'].mean()

print(f"Module 2 — Batch Test Results")
print()
print(f"  Overall accuracy  : {m2_acc:.4f} ({m2_df['correct'].sum()}/{len(m2_df)})")
print(f"  Mean confidence   : {m2_df['confidence'].mean():.4f}")
print(f"  High-risk flags   : {m2_df['risk_flag'].sum()} / {len(m2_df)}")
print()
print(m2_df.to_string(index = False))

Module 2 — Batch Test Results

  Overall accuracy  : 0.5238 (11/21)
  Mean confidence   : 0.8760
  High-risk flags   : 9 / 21

expected      pred  correct  confidence  risk_flag                                                              text
 sadness   sadness     True      0.9998       True i feel so empty and hopeless i cannot imagine things ever getting
 sadness      fear    False      0.9306       True              i cry every night and i do not even know why anymore
 sadness   sadness     True      0.9916       True                        i feel like a burden to everyone around me
 sadness   sadness     True      0.6362      False                 i have lost all interest in things i used to love
 sadness   sadness     True      1.0000       True                 nobody understands me and i feel completely alone
     joy  surprise    False      0.9885      False           i am so excited about my future and feel full of energy
     joy       joy     True      0.9995      False    

### 3.3 Failure Analysis

In [16]:
m2_failures = [(exp, text) for (exp, text), rec in zip(m2_test_cases, m2_records) if not rec['correct']]

if not m2_failures:
    print("No failures as all predictions correct")
else:
    print(f"Failures ({len(m2_failures)} total):\n")
    for expected, text in m2_failures:
        r = classify_emotion(text)
        print(f"  Expected   : {expected}")
        print(f"  Predicted  : {r['emotion']}  (confidence = {r['confidence']:.4f})")
        print(f"  Text       : {text[:70]}")
        print(f"  Instruction: {r['llm_instruction'][:90]}")
        print()

Failures (10 total):

  Expected   : sadness
  Predicted  : fear  (confidence = 0.9306)
  Text       : i cry every night and i do not even know why anymore
  Instruction: The user appears to be experiencing fear. Reassuring, grounding, structured. High-risk emo

  Expected   : joy
  Predicted  : surprise  (confidence = 0.9885)
  Text       : i am so excited about my future and feel full of energy
  Instruction: The user appears to be experiencing surprise. Curious, engaged, open. 

  Expected   : joy
  Predicted  : surprise  (confidence = 0.9926)
  Text       : i finally feel like things are coming together in my life
  Instruction: The user appears to be experiencing surprise. Curious, engaged, open. 

  Expected   : love
  Predicted  : surprise  (confidence = 0.7727)
  Text       : i feel so much love and warmth for the people in my life
  Instruction: The user appears to be experiencing surprise. Curious, engaged, open. 

  Expected   : love
  Predicted  : joy  (confidence = 0.6547)

## 4. Intent Classifier Module

Reconstructs the full classification engine from the saved artifacts in module_3_intent_classifier/artifacts/.  

The classify_intent() and get_routing_decision() functions are defined here identically to Module 3 so the pipeline notebook does not depend on Module 3 being in scope.

In [17]:
# Load all Module 3 artifacts
crisis_config = joblib.load(M3_ARTIFACT / 'crisis_config.joblib')

with open(M3_ARTIFACT / 'intent_schema.json', encoding = 'utf-8') as f:
    INTENT_SCHEMA = json.load(f)

with open(M3_ARTIFACT / 'few_shot_examples.json', encoding = 'utf-8') as f:
    FEW_SHOT_EXAMPLES = json.load(f)

with open(M3_ARTIFACT / 'system_prompt.txt', encoding = 'utf-8') as f:
    SYSTEM_PROMPT = f.read()

CRISIS_SIGNALS            = crisis_config['crisis_signals']
ISOLATION_PHRASES_M3      = crisis_config['isolation_phrases']
NEGATION_DISTRESS_PHRASES = crisis_config['negation_distress_phrases']
CLINICAL_DISTRESS_PHRASES = crisis_config['clinical_distress_phrases']

groq_client = Groq(api_key = GROQ_API_KEY)
M3_MODEL    = 'llama-3.1-8b-instant'

print(f"Intent schema     : {list(INTENT_SCHEMA.keys())}")
print(f"Few-shot examples : {len(FEW_SHOT_EXAMPLES)}")
print(f"Crisis signals    : {len(CRISIS_SIGNALS)}")
print(f"System prompt     : {len(SYSTEM_PROMPT):,} chars")
print(f"Groq model        : {M3_MODEL}")

Intent schema     : ['greeting', 'goodbye', 'gratitude', 'asking_mental_health_question', 'out_of_scope']
Few-shot examples : 43
Crisis signals    : 46
System prompt     : 17,134 chars
Groq model        : llama-3.1-8b-instant


In [18]:
def detect_crisis(text: str) -> dict:
    text_lower = text.lower()
    for signal in CRISIS_SIGNALS:
        if signal in text_lower:
            return {'crisis_detected': True, 'matched_signal': signal}
    return {'crisis_detected': False, 'matched_signal': None}


def classify_intent(
    text                 : str,
    conversation_history : Optional[list] = None,
    detected_emotion     : Optional[str]  = None,
    emotion_confidence   : Optional[float] = None,
    detected_language    : Optional[str]  = None
) -> dict:
    # Layer 1: Crisis detection
    crisis_result = detect_crisis(text)
    if crisis_result['crisis_detected']:
        return {
            'intent'           : 'asking_mental_health_question',
            'primary_intent'   : 'asking_mental_health_question',
            'secondary_intent' : None,
            'confidence'       : 'high',
            'routing'          : 'rag',
            'response_style'   : 'crisis_intervention',
            'reasoning'        : f"CRISIS DETECTED — matched signal: '{crisis_result['matched_signal']}'",
            'crisis_flag'      : True,
            'crisis_level'     : 'critical',
            'safety_layer'     : 'crisis_keyword_scan',
            'raw_message'      : text,
            'detected_emotion' : detected_emotion,
            'detected_language': detected_language
        }

    # Layer 2: LLM few-shot classification
    context_lines = []
    if detected_emotion and emotion_confidence:
        context_lines.append(f'Detected emotion: {detected_emotion} (confidence: {emotion_confidence:.2f})')
    if detected_language:
        context_lines.append(f'Detected language: {detected_language}')
    if conversation_history:
        history_str = ' | '.join(
            [f"{t['role']}: {t['content'][:60]}" for t in (conversation_history[-3:])]
        )
        context_lines.append(f'Recent history: {history_str}')

    context_block = ('\nContext:\n' + '\n'.join(context_lines)) if context_lines else ''
    user_message  = f'Classify the intent of this message:{context_block}\n\nMessage: {text}'

    try:
        response = groq_client.chat.completions.create(
            model    = M3_MODEL,
            messages = [
                {'role': 'system', 'content': SYSTEM_PROMPT},
                {'role': 'user',   'content': user_message}
            ],
            temperature     = 0.0,
            max_tokens      = 300,
            response_format = {'type': 'json_object'}
        )
        result = json.loads(response.choices[0].message.content.strip())
        result['crisis_flag']       = False
        result['crisis_level']      = None
        result['safety_layer']      = 'llm_classification'
        result['raw_message']       = text
        result['detected_emotion']  = detected_emotion
        result['detected_language'] = detected_language
        return result

    except (json.JSONDecodeError, Exception):
        return {
            'intent'           : 'asking_mental_health_question',
            'primary_intent'   : 'asking_mental_health_question',
            'secondary_intent' : None,
            'confidence'       : 'low',
            'routing'          : 'rag',
            'response_style'   : 'empathetic_support',
            'reasoning'        : 'fallback — JSON parse error or API failure',
            'crisis_flag'      : False,
            'crisis_level'     : None,
            'safety_layer'     : 'fallback',
            'raw_message'      : text,
            'detected_emotion' : detected_emotion,
            'detected_language': detected_language
        }


def get_routing_decision(
    text                 : str,
    conversation_history : Optional[list] = None,
    detected_emotion     : Optional[str]  = None,
    emotion_confidence   : Optional[float] = None,
    detected_language    : Optional[str]  = None
) -> dict:
    result = classify_intent(
        text                 = text,
        conversation_history = conversation_history,
        detected_emotion     = detected_emotion,
        emotion_confidence   = emotion_confidence,
        detected_language    = detected_language
    )
    return {
        'routing'          : result.get('routing', 'rag'),
        'intent'           : result.get('intent', 'asking_mental_health_question'),
        'primary_intent'   : result.get('primary_intent', 'asking_mental_health_question'),
        'secondary_intent' : result.get('secondary_intent'),
        'confidence'       : result.get('confidence', 'low'),
        'crisis_flag'      : result.get('crisis_flag', False),
        'crisis_level'     : result.get('crisis_level'),
        'response_style'   : result.get('response_style', 'empathetic_support'),
        'reasoning'        : result.get('reasoning', ''),
        'safety_layer'     : result.get('safety_layer', ''),
        'rag_needed'       : result.get('routing') == 'rag',
        'raw_message'      : text,
        'detected_emotion' : detected_emotion,
        'detected_language': detected_language
    }


print("classify_intent() and get_routing_decision() ready")

classify_intent() and get_routing_decision() ready


### 4.1 Functional Smoke Test

One sample per intent class to confirm the LLM is responding and JSON parsing is working.

In [19]:
smoke_intents = [
    ('greeting',                    'hello, i am glad to be here today'),
    ('goodbye',                     'thank you so much, goodbye'),
    ('gratitude',                   'i really appreciate all your help'),
    ('asking_mental_health_question','i have been struggling with anxiety for weeks and cannot sleep'),
    ('out_of_scope',                'what is the capital of france'),
    ('asking_mental_health_question','i want to kill myself'),    # crisis -> overrides to MH
]

print("Smoke Test\n")
for expected, text in smoke_intents:
    d      = get_routing_decision(text, detected_emotion = 'sadness', emotion_confidence = 0.90, detected_language = 'en')
    status = 'OK' if d['primary_intent'] == expected else 'FAIL'
    crisis = ' [CRISIS]' if d['crisis_flag'] else ''
    print(f"  [{status}] Expected: {expected:<35} | Pred: {d['primary_intent']:<35} | Routing: {d['routing']}{crisis}")
    print(f"         Text    : {text[:70]}")
    print()

Smoke Test

  [OK] Expected: greeting                            | Pred: greeting                            | Routing: direct
         Text    : hello, i am glad to be here today

  [OK] Expected: goodbye                             | Pred: goodbye                             | Routing: rag
         Text    : thank you so much, goodbye

  [OK] Expected: gratitude                           | Pred: gratitude                           | Routing: direct
         Text    : i really appreciate all your help

  [OK] Expected: asking_mental_health_question       | Pred: asking_mental_health_question       | Routing: rag
         Text    : i have been struggling with anxiety for weeks and cannot sleep

  [OK] Expected: out_of_scope                        | Pred: out_of_scope                        | Routing: decline
         Text    : what is the capital of france

  [OK] Expected: asking_mental_health_question       | Pred: asking_mental_health_question       | Routing: rag [CRISIS]
         

### 4.2 Full Batch Test

Covers all five intent classes, crisis escalation, multilingual routing, compound intents and edge cases.

In [20]:
# Each entry: (expected_intent, text, detected_emotion, emotion_confidence, detected_language)
m3_test_cases = [
    # Greetings
    ('greeting', 'hello, i am new here and looking forward to talking', 'joy',     0.80, 'en'),
    ('greeting', 'hi there, good morning', 'joy',     0.70, 'en'),
    ('greeting', 'مرحبا، أنا سعيد بالتحدث معك', 'joy',     0.72, 'ar'),

    # Goodbye
    ('goodbye', 'thank you so much, take care, goodbye', 'joy',     0.85, 'en'),
    ('goodbye', 'bye for now, i will talk to you later', 'joy',     0.75, 'en'),

    # Gratitude
    ('gratitude', 'thank you that was really helpful',             'joy',     0.88, 'en'),
    ('gratitude', 'i really appreciate everything you have done',  'love',    0.91, 'en'),
    ('gratitude', 'merci beaucoup ça m aide vraiment',             'joy',     0.84, 'fr'),

    # Mental health questions
    ('asking_mental_health_question', 'i have been struggling with severe anxiety for months',      'fear',    0.95, 'en'),
    ('asking_mental_health_question', 'i feel depressed and empty every single day',                'sadness', 0.97, 'en'),
    ('asking_mental_health_question', 'how do i cope with panic attacks at work',                   'fear',    0.90, 'en'),
    ('asking_mental_health_question', 'i feel like a burden to my family',                          'sadness', 0.93, 'en'),
    ('asking_mental_health_question', 'ich fühle mich sehr deprimiert und weiß nicht weiter',       'sadness', 0.94, 'de'),
    ('asking_mental_health_question', 'اشعر بالقلق الشديد ولا اعرف ماذا افعل',                    'fear',    0.91, 'ar'),
    ('asking_mental_health_question', '最近ずっと不安で眠れない夜が続いています',                        'fear',    0.93, 'ja'),

    # Out of scope
    ('out_of_scope', 'what is the capital of france',                'surprise', 0.60, 'en'),
    ('out_of_scope', 'can you write me a python function',           'surprise', 0.55, 'en'),
    ('out_of_scope', 'what is the weather like today',               'surprise', 0.58, 'en'),

    # Crisis escalation — must route to MH regardless of text surface form
    ('asking_mental_health_question', 'i want to kill myself',                             'sadness', 0.99, 'en'),
    ('asking_mental_health_question', 'i have been thinking about ending my life',         'sadness', 0.99, 'en'),
    ('asking_mental_health_question', 'goodbye forever i cannot do this anymore',          'sadness', 0.98, 'en'),
    ('asking_mental_health_question', 'ich will sterben',                                  'sadness', 0.99, 'de'),
    ('asking_mental_health_question', 'je veux mourir tellement je souffre',               'sadness', 0.99, 'fr'),

    # Compound intents — gratitude + MH content
    ('asking_mental_health_question', 'thank you but i still feel completely empty inside', 'sadness', 0.88, 'en'),
    ('asking_mental_health_question', 'thanks but the anxiety is still very bad',           'fear',    0.87, 'en'),
]

m3_records = []
for expected, text, emotion, conf, lang in m3_test_cases:
    t0 = time.time()
    d  = get_routing_decision(
        text               = text,
        detected_emotion   = emotion,
        emotion_confidence = conf,
        detected_language  = lang
    )
    latency = (time.time() - t0) * 1000
    m3_records.append({
        'expected'     : expected,
        'pred'         : d['primary_intent'],
        'correct'      : d['primary_intent'] == expected,
        'routing'      : d['routing'],
        'crisis_flag'  : d['crisis_flag'],
        'confidence'   : d['confidence'],
        'safety_layer' : d['safety_layer'],
        'latency_ms'   : round(latency, 1),
        'text'         : text[:60]
    })

m3_df = pd.DataFrame(m3_records)

m3_acc_all    = m3_df['correct'].mean()
m3_acc_mh     = m3_df[m3_df['expected'] == 'asking_mental_health_question']['correct'].mean()
m3_acc_crisis = m3_df[m3_df['crisis_flag']]['correct'].mean() if m3_df['crisis_flag'].any() else float('nan')
m3_avg_lat    = m3_df['latency_ms'].mean()

print(f"Batch Test Results")
print()
print(f"  Overall accuracy      : {m3_acc_all:.4f} ({m3_df['correct'].sum()}/{len(m3_df)})")
print(f"  MH question accuracy  : {m3_acc_mh:.4f}")
print(f"  Crisis cases correct  : {m3_df[m3_df['crisis_flag']]['correct'].sum()} / {m3_df['crisis_flag'].sum()}")
print(f"  Mean latency          : {m3_avg_lat:.0f} ms")
print()
print(m3_df.to_string(index = False))

Module 3 — Batch Test Results

  Overall accuracy      : 0.9600 (24/25)
  MH question accuracy  : 1.0000
  Crisis cases correct  : 2 / 2
  Mean latency          : 36094 ms

                     expected                          pred  correct routing  crisis_flag confidence        safety_layer  latency_ms                                                  text
                     greeting                      greeting     True  direct        False       high  llm_classification     34176.9   hello, i am new here and looking forward to talking
                     greeting                      greeting     True  direct        False       high  llm_classification     41721.7                                hi there, good morning
                     greeting                      greeting     True  direct        False       high  llm_classification     42709.6                           مرحبا، أنا سعيد بالتحدث معك
                      goodbye                     gratitude    False  direct   

### 4.3 Failure Analysis

In [21]:
m3_failures = [(exp, text, em, cf, lg) for (exp, text, em, cf, lg), rec in zip(m3_test_cases, m3_records) if not rec['correct']]

if not m3_failures:
    print("No failures — all predictions correct.")
else:
    print(f"Failures ({len(m3_failures)} total):\n")
    for expected, text, emotion, conf, lang in m3_failures:
        d = get_routing_decision(text, detected_emotion = emotion, emotion_confidence = conf, detected_language = lang)
        print(f"  Expected   : {expected}")
        print(f"  Predicted  : {d['primary_intent']}  (confidence = {d['confidence']})")
        print(f"  Routing    : {d['routing']}  |  Safety layer: {d['safety_layer']}")
        print(f"  Reasoning  : {d['reasoning'][:90]}")
        print(f"  Text       : {text[:70]}")
        print()

Failures (1 total):

  Expected   : goodbye
  Predicted  : gratitude  (confidence = high)
  Routing    : direct  |  Safety layer: llm_classification
  Reasoning  : Gratitude expression followed by goodbye — gratitude takes precedence, warm acknowledgment
  Text       : thank you so much, take care, goodbye



## 5. Full Pipeline Simulation

Each test input flows through all three modules in the correct production order:

```
text  →  Module 1 (language)  →  Module 2 (emotion)  →  Module 3 (intent + routing)
```

This is the exact call sequence Module 4 RAG and the orchestrator will use.

In [22]:
def run_pipeline(text: str, conversation_history: Optional[list] = None) -> dict:
    # Module 1
    lang_result = detect_language(text, m1_model, m1_vectorizer)

    # Module 2
    emotion_result = classify_emotion(text)

    # Module 3 (intent + routing)
    routing_result = get_routing_decision(
        text                 = text,
        conversation_history = conversation_history,
        detected_emotion     = emotion_result['emotion'],
        emotion_confidence   = emotion_result['confidence'],
        detected_language    = lang_result['prediction']
    )

    return {
        'text'             : text,
        'language'         : lang_result['prediction'],
        'lang_name'        : lang_result['lang_name'],
        'lang_confidence'  : lang_result['confidence'],
        'lang_trusted'     : lang_result['trusted'],
        'emotion'          : emotion_result['emotion'],
        'emotion_conf'     : emotion_result['confidence'],
        'risk_flag'        : emotion_result['risk_flag'],
        'llm_instruction'  : emotion_result['llm_instruction'],
        'intent'           : routing_result['primary_intent'],
        'routing'          : routing_result['routing'],
        'crisis_flag'      : routing_result['crisis_flag'],
        'response_style'   : routing_result['response_style'],
        'reasoning'        : routing_result['reasoning'],
        'safety_layer'     : routing_result['safety_layer']
    }


print("run_pipeline() ready")

run_pipeline() ready


### 5.1 Full Pipeline Batch Test

Representative inputs spanning all languages, emotion classes, routing paths and crisis escalation scenarios.

In [23]:
pipeline_cases = [
    # English
    ('en', 'I have been struggling with severe anxiety and cannot sleep at night'),
    ('en', 'I feel so depressed and empty, I have lost all motivation'),
    ('en', 'hello, I am glad to be here and looking forward to getting support'),
    ('en', 'thank you so much, this session really helped me today'),
    ('en', 'goodbye, I will talk to you next week'),
    ('en', 'what is the capital of france'),

    # crisis cases
    ('en', 'I want to kill myself, I cannot take this pain anymore'),
    ('en', 'I have been thinking about ending my life for weeks'),
    ('en', 'goodbye forever, I am done with everything'),

    # compound and edge cases
    ('en', 'thank you but I still feel completely empty inside'),
    ('en', 'nobody understands me and I feel completely alone'),

    # Arabic
    ('ar', 'أشعر بقلق شديد ولا أستطيع النوم في الليل'),
    ('ar', 'مرحبا، أنا هنا للحصول على المساعدة'),
    ('ar', 'أريد إنهاء حياتي'),    # crisis

    # German
    ('de', 'ich fühle mich sehr deprimiert und weiß nicht mehr weiter'),
    ('de', 'danke, aber ich fühle mich immer noch sehr leer'),
    ('de', 'ich will sterben'),    # crisis

    # French
    ('fr', 'je me sens très anxieux et je ne sais pas quoi faire'),
    ('fr', 'merci beaucoup, ça m a vraiment aidé'),
    ('fr', 'je veux mourir tellement je souffre'),    # crisis

    # Spanish
    ('es', 'estoy luchando con ansiedad severa y no puedo dormir'),

    # Chinese
    ('zh', '我最近感到非常焦虑，不知道该如何应对'),
    ('zh', '我想自杀'),    # crisis

    # Japanese
    ('ja', '最近ずっと不安で、夜も眠れない日が続いています'),
    ('ja', '死にたい'),    # crisis
]

In [24]:
pipeline_records = []
for expected_lang, text in pipeline_cases:
    t0 = time.time()
    r  = run_pipeline(text)
    latency = (time.time() - t0) * 1000
    pipeline_records.append({
        'true_lang'    : expected_lang,
        'pred_lang'    : r['language'],
        'lang_ok'      : r['language'] == expected_lang,
        'emotion'      : r['emotion'],
        'risk_flag'    : r['risk_flag'],
        'intent'       : r['intent'],
        'routing'      : r['routing'],
        'crisis_flag'  : r['crisis_flag'],
        'response_style': r['response_style'],
        'latency_ms'   : round(latency, 1),
        'text'         : text[:55]
    })

pipeline_df = pd.DataFrame(pipeline_records)

lang_acc   = pipeline_df['lang_ok'].mean()
crisis_cnt = pipeline_df['crisis_flag'].sum()
rag_cnt    = (pipeline_df['routing'] == 'rag').sum()
direct_cnt = (pipeline_df['routing'] == 'direct').sum()
avg_lat    = pipeline_df['latency_ms'].mean()

print(f"Batch Test Results")
print()
print(f"  Language accuracy : {lang_acc:.4f} ({pipeline_df['lang_ok'].sum()}/{len(pipeline_df)})")
print(f"  Routing → RAG     : {rag_cnt}/{len(pipeline_df)}")
print(f"  Routing → Direct  : {direct_cnt}/{len(pipeline_df)}")
print(f"  Crisis flags      : {crisis_cnt}/{len(pipeline_df)}")
print(f"  Mean latency      : {avg_lat:.0f} ms")
print()
print(pipeline_df.to_string(index = False))

Batch Test Results

  Language accuracy : 0.9200 (23/25)
  Routing → RAG     : 21/25
  Routing → Direct  : 3/25
  Crisis flags      : 2/25
  Mean latency      : 20740 ms

true_lang pred_lang  lang_ok   emotion  risk_flag                        intent routing  crisis_flag                      response_style  latency_ms                                                    text
       en        en     True      fear       True asking_mental_health_question     rag        False                  empathetic_support       878.7 I have been struggling with severe anxiety and cannot s
       en        en     True   sadness       True asking_mental_health_question     rag        False                  empathetic_support     36199.8 I feel so depressed and empty, I have lost all motivati
       en        en     True       joy      False                      greeting  direct        False                 warm_acknowledgment     26001.9 hello, I am glad to be here and looking forward to gett
       en

### 5.2 Detailed Pipeline Output

Prints the full output dictionary for each test case so the data flow from language → emotion → routing is fully visible.

In [25]:
print("Full Pipeline — Detailed Output\n")
for expected_lang, text in pipeline_cases:
    r = run_pipeline(text)
    crisis = ' CRISIS' if r['crisis_flag'] else ''
    risk   = ' RISK'   if r['risk_flag']   else ''
    print(f"  Input    : {text[:70]}")
    print(f"  Language : {r['language']} ({r['lang_name']})  conf = {r['lang_confidence']:.3f}  trusted = {r['lang_trusted']}")
    print(f"  Emotion  : {r['emotion']}  conf = {r['emotion_conf']:.3f}{risk}")
    print(f"  Intent   : {r['intent']}  routing = {r['routing']}{crisis}")
    print(f"  Style    : {r['response_style']}")
    print(f"  Reason   : {r['reasoning'][:85]}")
    print()

Full Pipeline — Detailed Output

  Input    : I have been struggling with severe anxiety and cannot sleep at night
  Language : en (English)  conf = 0.963  trusted = True
  Emotion  : fear  conf = 0.992 RISK
  Intent   : asking_mental_health_question  routing = rag
  Style    : empathetic_support
  Reason   : Explicit expression of severe anxiety and sleep disturbance — core mental health conc

  Input    : I feel so depressed and empty, I have lost all motivation
  Language : en (English)  conf = 0.989  trusted = True
  Emotion  : sadness  conf = 1.000 RISK
  Intent   : asking_mental_health_question  routing = rag
  Style    : empathetic_support
  Reason   : Explicit expression of depression and emptiness, loss of motivation is a significant 

  Input    : hello, I am glad to be here and looking forward to getting support
  Language : en (English)  conf = 0.980  trusted = True
  Emotion  : joy  conf = 0.997
  Intent   : asking_mental_health_question  routing = rag
  Style    : empathe

## 6. Integration Test Summary

Aggregates the results from all three modules and the full pipeline run into a single readiness assessment.

In [26]:
print("Integration Test Summary")
print()
print(f"  Module 1 — Language Detection")
print(f"    Batch accuracy (overall) : {m1_acc_overall:.4f}")
print(f"    Batch accuracy (long)    : {m1_acc_long:.4f}")
print(f"    Batch accuracy (short)   : {m1_acc_short:.4f}")
print(f"    Mean confidence          : {m1_df['confidence'].mean():.4f}")
print()
print(f"  Module 2 — Emotion Classifier")
print(f"    Batch accuracy           : {m2_acc:.4f}")
print(f"    Mean confidence          : {m2_df['confidence'].mean():.4f}")
print(f"    High-risk flags fired    : {m2_df['risk_flag'].sum()} / {len(m2_df)}")
print()
print(f"  Module 3 — Intent Classifier")
print(f"    Batch accuracy (overall) : {m3_acc_all:.4f}")
print(f"    MH question accuracy     : {m3_acc_mh:.4f}")
print(f"    Crisis cases correct     : {m3_df[m3_df['crisis_flag']]['correct'].sum()} / {m3_df['crisis_flag'].sum()}")
print(f"    Mean latency             : {m3_avg_lat:.0f} ms")
print()
print(f"  Full Pipeline")
print(f"    Language accuracy        : {lang_acc:.4f}")
print(f"    RAG routing              : {rag_cnt}/{len(pipeline_df)}")
print(f"    Direct routing           : {direct_cnt}/{len(pipeline_df)}")
print(f"    Crisis escalations       : {crisis_cnt}/{len(pipeline_df)}")
print(f"    Mean end-to-end latency  : {avg_lat:.0f} ms")
print()

# Readiness gate
m1_pass = m1_acc_overall >= 0.90
m2_pass = m2_acc         >= 0.80
m3_pass = m3_acc_all     >= 0.80
all_pass = m1_pass and m2_pass and m3_pass

print(f"  Readiness Gates (thresholds: M1 >= 0.90 | M2 >= 0.80 | M3 >= 0.80)")
print(f"    Module 1 : {'PASS' if m1_pass else 'FAIL'}")
print(f"    Module 2 : {'PASS' if m2_pass else 'FAIL'}")
print(f"    Module 3 : {'PASS' if m3_pass else 'FAIL'}")
print()
print(f"  Pipeline status : {'READY for Module 4 RAG' if all_pass else 'NOT READY — fix failing modules before proceeding'}")

Integration Test Summary

  Module 1 — Language Detection
    Batch accuracy (overall) : 1.0000
    Batch accuracy (long)    : 1.0000
    Batch accuracy (short)   : 1.0000
    Mean confidence          : 0.9723

  Module 2 — Emotion Classifier
    Batch accuracy           : 0.5238
    Mean confidence          : 0.8760
    High-risk flags fired    : 9 / 21

  Module 3 — Intent Classifier
    Batch accuracy (overall) : 0.9600
    MH question accuracy     : 1.0000
    Crisis cases correct     : 2 / 2
    Mean latency             : 36094 ms

  Full Pipeline
    Language accuracy        : 0.9200
    RAG routing              : 21/25
    Direct routing           : 3/25
    Crisis escalations       : 2/25
    Mean end-to-end latency  : 20740 ms

  Readiness Gates (thresholds: M1 >= 0.90 | M2 >= 0.80 | M3 >= 0.80)
    Module 1 : PASS
    Module 2 : FAIL
    Module 3 : PASS

  Pipeline status : NOT READY — fix failing modules before proceeding
